In [1]:
import arcpy

In [ ]:
# ── LOAD DATA FOR MISSION ────────────────────────────────────────────────────
points = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\arcgis_UVI2_Around_STX_20250306\arcgis_UVI2_Around_STX_20250306.gdb\calc_humpback_points_NAD83"
shore = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\CARIBBEAN_SHORELINES"

# nearest shoreline calculation columns
fld_dist = "SHORE_DISTANCE_m"
fld_x    = "near_shore_x_coord"
fld_y    = "near_shore_y_coord"
fld_id   = "near_shore_id"

SEARCH_RADIUS = None

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True

In [ ]:
# ── HELPER (add field only if it does NOT already exist) ─────────────────────
def add_field_if_missing(table, field_name, field_type, field_length=None):
    existing = {f.name for f in arcpy.ListFields(table)}
    if field_name not in existing:
        kwargs = {"field_type": field_type}
        if field_type.upper() == "TEXT" and field_length:
            kwargs["field_length"] = field_length
        arcpy.management.AddField(table, field_name, **kwargs)

In [ ]:
# ── 1. RUN NEAR TOOL ─────────────────────────────────────────────────────────
arcpy.analysis.Near(
    in_features=points,
    near_features=shore,
    search_radius=SEARCH_RADIUS,
    location="LOCATION",
    angle="NO_ANGLE",
    method="GEODESIC",
    distance_unit="Meters"
)

print("Near tool complete.")

In [ ]:
# ── 2. ADD CUSTOM FIELDS ─────────────────────────────────────────────────────
for name, ftype in [(fld_dist, "DOUBLE"), (fld_x, "DOUBLE"),
                    (fld_y, "DOUBLE"), (fld_id, "LONG")]:
    add_field_if_missing(points, name, ftype)
    
print("Fields ready.")

In [ ]:
# ── 3. COPY NEAR → CUSTOM FIELDS ─────────────────────────────────────────────
copy_fields = ["NEAR_DIST", "NEAR_X", "NEAR_Y", "NEAR_FID",
                fld_dist, fld_x, fld_y, fld_id]

with arcpy.da.UpdateCursor(points, copy_fields) as cursor:
    for row in cursor:
        row[4], row[5], row[6], row[7] = row[0], row[1], row[2], row[3]
        cursor.updateRow(row)
        
print("Results copied to custom fields.")

In [ ]:
# ── 4. DROP DEFAULT NEAR_* FIELDS ────────────────────────────────────────────
existing = {f.name for f in arcpy.ListFields(points)}
to_drop  = [f for f in ["NEAR_DIST", "NEAR_X", "NEAR_Y", "NEAR_FID"]
            if f in existing]
if to_drop:
    arcpy.management.DeleteField(points, to_drop)
    print(f"Dropped: {to_drop}")
    
print("Done.")